In [7]:
import requests, sqlite3, pandas as pd

url = "https://raw.githubusercontent.com/davidjamesknight/SQLite_databases_for_learning_data_science/main/diamonds.db"
r = requests.get(url)

with open("diamonds.db", "wb") as f:
    f.write(r.content)

conn = sqlite3.connect("diamonds.db")

query = """
SELECT
  O.carat,
  O.price,
  O.depth,
  "O"."table",
  O.x,
  O.y,
  O.z,
  C.cut,
  Co.color,
  Cl.clarity
FROM
  Observation AS O
JOIN
  Cut AS C ON O.cut_id = C.cut_id
JOIN
  Color AS Co ON O.color_id = Co.color_id
JOIN
  Clarity AS Cl ON O.clarity_id = Cl.clarity_id
"""

df = pd.read_sql_query(query, conn)
df.head()

,carat,price,depth,table,x,y,z,cut,color,clarity
0,0.23,326,61.5,55.0,3.95,3.98,2.43,Ideal,E,SI2
1,0.21,326,59.8,61.0,3.89,3.84,2.31,Premium,E,SI1
2,0.23,327,56.9,65.0,4.05,4.07,2.31,Good,E,VS1
3,0.29,334,62.4,58.0,4.20,4.23,2.63,Premium,I,VS2
4,0.31,335,63.3,58.0,4.34,4.35,2.75,Good,J,SI2


## 📋 Recap del Análisis Exploratorio

En el notebook anterior exploramos el dataset de **53,940 diamantes**. Aquí un resumen rápido:

### Variables
| Tipo | Variables |
|---|---|
| **Numéricas** | `carat`, `depth`, `table`, `x`, `y`, `z`, `price` |
| **Categóricas (ordinales)** | `cut` (Fair → Premium), `color` (D → J), `clarity` (IF → I1) |

### Hallazgos clave
- 💎 **`carat`**, **`x`**, **`y`** y **`z`** tienen la correlación más fuerte con `price`
- ⚠️ `depth` y `table` muestran alta dispersión y outliers elevados
- 📊 `price` y `carat` tienen distribuciones **sesgadas a la derecha**
- 🔗 Las variables de dimensión (`x`, `y`, `z`) están **altamente correlacionadas entre sí** → posible multicolinealidad

### Variable objetivo
> Predeciremos **`price`** (precio en USD) usando las demás características del diamante.

## 1️⃣ Preprocesamiento

Antes de ajustar el modelo, preparamos los datos:
- Separamos la variable objetivo (`price`) de las variables predictoras
- Dividimos en **80% entrenamiento / 20% prueba**

In [8]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder

# Separamos predictores y variable objetivo
X = df.drop(columns=["price"]).copy()
y = df["price"].copy()

# Definimos columnas categóricas y numéricas
cat_cols = ["cut", "color", "clarity"]
num_cols = [c for c in X.columns if c not in cat_cols]

# Partición 80/20
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print(f"X_train: {X_train.shape} | X_test: {X_test.shape}")
print(f"y_train: {y_train.shape} | y_test: {y_test.shape}")

X_train: (43152, 9) | X_test: (10788, 9)
y_train: (43152,) | y_test: (10788,)


### Encoding de variables categóricas

Las variables `cut`, `color` y `clarity` son categóricas ordinales.  
Usamos **OrdinalEncoder** respetando el orden natural de cada una.

In [9]:
# Ordenes ordinales definidos segun la calidad natural
cut_order = ["Fair", "Good", "Very Good", "Premium", "Ideal"]
color_order = ["J", "I", "H", "G", "F", "E", "D"]
clarity_order = ["I1", "SI2", "SI1", "VS2", "VS1", "VVS2", "VVS1", "IF"]

encoder = OrdinalEncoder(
    categories=[cut_order, color_order, clarity_order],
    dtype=float
)

# Ajustamos solo con train para evitar leakage
X_train_encoded = X_train.copy()
X_test_encoded = X_test.copy()

X_train_encoded[cat_cols] = encoder.fit_transform(X_train[cat_cols])
X_test_encoded[cat_cols] = encoder.transform(X_test[cat_cols])

X_train_encoded.head()

,carat,depth,table,x,y,z,cut,color,clarity
26546,2.01,58.1,64.0,8.23,8.19,4.77,1.0,4.0,1.0
9159,1.01,60.0,60.0,6.57,6.49,3.92,2.0,5.0,1.0
14131,1.10,62.5,58.0,6.59,6.54,4.10,3.0,2.0,3.0
15757,1.50,61.5,65.0,7.21,7.17,4.42,1.0,5.0,1.0
24632,1.52,62.1,57.0,7.27,7.32,4.53,2.0,3.0,4.0


## 2️⃣ Ajuste del Modelo — Regresión Lineal (OLS)

Ajustamos un modelo **crudo** con todas las variables, sin ningún tipo de selección.  
Usamos `statsmodels` para ver el resumen estadístico completo.

In [10]:
import statsmodels.api as sm

# Agregamos termino independiente (intercepto)
X_train_sm = sm.add_constant(X_train_encoded, has_constant="add")

ols_model = sm.OLS(y_train, X_train_sm).fit()
print(ols_model.summary())

                            OLS Regression Results                            
Dep. Variable:                  price   R-squared:                       0.907
Model:                            OLS   Adj. R-squared:                  0.907
Method:                 Least Squares   F-statistic:                 4.694e+04
Date:                Sun, 08 Mar 2026   Prob (F-statistic):               0.00
Time:                        20:37:01   Log-Likelihood:            -3.6770e+05
No. Observations:               43152   AIC:                         7.354e+05
Df Residuals:                   43142   BIC:                         7.355e+05
Df Model:                           9                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const       3830.8354    472.330      8.110      0.0

## 3️⃣ Evaluación en Test

Medimos el desempeño del modelo con datos que **nunca vio durante el entrenamiento**.

In [11]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

X_test_sm = sm.add_constant(X_test_encoded, has_constant="add")
y_pred = ols_model.predict(X_test_sm)

mae = mean_absolute_error(y_test, y_pred)
rmse = mean_squared_error(y_test, y_pred) ** 0.5
r2 = r2_score(y_test, y_pred)

print(f"MAE : {mae:,.2f}")
print(f"RMSE: {rmse:,.2f}")
print(f"R2  : {r2:.4f}")

results = pd.DataFrame({
    "y_real": y_test.values,
    "y_pred": y_pred.values
})
results.head()

MAE : 805.27
RMSE: 1,224.60
R2  : 0.9057


,y_real,y_pred
0,559,933.809101
1,2201,3051.529142
2,1238,2150.096513
3,1304,2264.065966
4,6901,10212.237276
